In [239]:
import os, json, sqlite3
import numpy as np
import pandas as pd
#import anthropic
from langchain.agents import Agent, create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from openai import OpenAI
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

In [240]:
from dotenv import load_dotenv
load_dotenv()

True

In [241]:
#os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [242]:
#llm = ChatOpenAI(model = "gpt-4o-mini", temperature= 0.2, max_tokens= 10)

def call_gpt4o(system_prompt: str, user_message: str) -> str:
    #using gpt-4o-mini
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        max_tokens=300,
        temperature=0.7,
    )
    return response.choices[0].message.content


In [243]:
def init_db():
    conn = sqlite3.connect("diseases.db")
    c = conn.cursor()

    c.execute("""CREATE TABLE IF NOT EXISTS diseases (
        id TEXT PRIMARY KEY, 
        name TEXT, 
        description TEXT,
        severity TEXT, 
        treatment TEXT, 
        prevention TEXT, 
        see_doctor TEXT
        )
              """) # disease table
    
    c.execute("""
    CREATE TABLE IF NOT EXISTS symptoms (
        id TEXT PRIMARY KEY
    )
    """) #symptoms table

    c.execute("""
    CREATE TABLE IF NOT EXISTS disease_symptoms (
        disease_id TEXT,
        symptom_id TEXT,
        PRIMARY KEY (disease_id, symptom_id),
        FOREIGN KEY (disease_id) REFERENCES diseases(id),
        FOREIGN KEY (symptom_id) REFERENCES symptoms(id)
    )
    """) # mapping table


    diseases = [
        ("flu", "Influenza", "Contagious respiratory illness", "Moderate",
         "Rest, antivirals, fever reducers", "Annual vaccine, hand hygiene",
         "If symptoms >7 days or difficulty breathing"),
        ("common_cold", "Common Cold", "Mild viral URI", "Mild",
         "Rest, decongestants, gargle", "Hand washing, sleep",
         "If symptoms persist >10 days"),
        ("covid19", "COVID-19", "SARS-CoV-2 illness", "Moderate-Severe",
         "Isolation, fever mgmt, antivirals", "Vaccination, masks",
         "If difficulty breathing or chest pain"),
        ("allergies", "Seasonal Allergies", "Immune reaction", "Mild",
         "Antihistamines, nasal sprays", "Monitor pollen",
         "If OTC meds don't help"),
        ("migraine", "Migraine", "Neurological headaches", "Moderate-Severe",
         "Triptans, dark room rest", "Identify triggers, regular sleep",
         "If frequency increases"),
        ("gastroenteritis", "Gastroenteritis", "Stomach inflammation",
         "Mild-Moderate", "ORS, BRAT diet", "Hand washing, safe food",
         "If can't keep fluids down 24h"),
        ("pneumonia", "Pneumonia", "Lung infection", "Severe",
         "Antibiotics, hospitalization", "Vaccines, hygiene",
         "Seek immediate care"),
        ("bronchitis", "Acute Bronchitis", "Bronchial inflammation",
         "Moderate", "Rest, cough suppressants", "Avoid smoking",
         "If cough >3 weeks"),
        ("sinusitis", "Sinus Infection", "Inflammation of sinuses", "Mild-Moderate",
        "Decongestants, steam inhalation", "Avoid allergens, stay hydrated",
        "If symptoms last >10 days"),
        ("viral_fever", "Viral Fever", "General viral infection causing fever", "Mild-Moderate",
        "Rest, fluids, fever reducers", "Hygiene, avoid contact",
        "If fever persists >5 days"),
        ("dehydration", "Dehydration", "Lack of sufficient body fluids", "Moderate",
        "ORS, fluids, electrolytes", "Drink enough fluids",
        "If dizziness or no urination"),
        ("asthma_attack", "Asthma Attack", "Airway inflammation causing breathing issues", "Severe",
        "Inhalers, bronchodilators", "Avoid triggers, regular medication",
        "Seek immediate care if breathing difficulty"),
        ("appendicitis", "Appendicitis", "Inflammation of appendix", "Severe",
        "Surgery (appendectomy)", "No specific prevention",
        "Immediate care if severe abdominal pain"),
    ]
    c.executemany(
        "INSERT OR REPLACE INTO diseases VALUES (?,?,?,?,?,?,?)", diseases
    )


    symptoms = [
        "fever","cough","headache","nausea","vomiting","fatigue",
        "sore_throat","chills","body_pain","loss_of_appetite",
        "abdominal_pain","diarrhea","sweating","rapid_breathing","dizziness"
    ]

    c.executemany(
        "INSERT OR IGNORE INTO symptoms VALUES (?)",
        [(s,) for s in symptoms]
    )


    disease_symptoms = {
        "flu": ["fever", "cough", "body_pain", "fatigue"],
        "common_cold": ["cough", "sore_throat"],
        "covid19": ["fever", "cough", "fatigue"],
        "allergies": ["cough", "sore_throat"],
        "migraine": ["headache", "nausea"],
        "gastroenteritis": ["nausea", "vomiting", "diarrhea"],
        "pneumonia": ["fever", "cough", "rapid_breathing"],
        "bronchitis": ["cough", "fatigue"],

        "sinusitis": ["headache", "sore_throat"],
        "viral_fever": ["fever", "fatigue"],
        "dehydration": ["dizziness", "fatigue"],
        "asthma_attack": ["rapid_breathing", "cough"],
        "appendicitis": ["abdominal_pain", "nausea"],
    }

    mapping_rows = [
        (disease, symptom)
        for disease, sym_list in disease_symptoms.items()
        for symptom in sym_list
    ]

    c.executemany(
        "INSERT OR IGNORE INTO disease_symptoms VALUES (?, ?)",
        mapping_rows
    )
    conn.commit()
    return conn

In [244]:

conn = sqlite3.connect("diseases.db")
c = conn.cursor()

c.execute("SELECT * FROM diseases")
rows = c.fetchall()

for row in rows:
    print(row)

conn.close()

('flu', 'Influenza', 'Contagious respiratory illness', 'Moderate', 'Rest, antivirals, fever reducers', 'Annual vaccine, hand hygiene', 'If symptoms >7 days or difficulty breathing')
('common_cold', 'Common Cold', 'Mild viral URI', 'Mild', 'Rest, decongestants, gargle', 'Hand washing, sleep', 'If symptoms persist >10 days')
('covid19', 'COVID-19', 'SARS-CoV-2 illness', 'Moderate-Severe', 'Isolation, fever mgmt, antivirals', 'Vaccination, masks', 'If difficulty breathing or chest pain')
('allergies', 'Seasonal Allergies', 'Immune reaction', 'Mild', 'Antihistamines, nasal sprays', 'Monitor pollen', "If OTC meds don't help")
('migraine', 'Migraine', 'Neurological headaches', 'Moderate-Severe', 'Triptans, dark room rest', 'Identify triggers, regular sleep', 'If frequency increases')
('gastroenteritis', 'Gastroenteritis', 'Stomach inflammation', 'Mild-Moderate', 'ORS, BRAT diet', 'Hand washing, safe food', "If can't keep fluids down 24h")
('pneumonia', 'Pneumonia', 'Lung infection', 'Severe'

In [245]:
import sys
from pathlib import Path

def _backend_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "app.py").exists():
        return here
    if (here.parent / "app.py").exists():
        return here.parent
    return here

_root = _backend_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.dp_train_utils import SYMPTOMS, load_dp_frame, train_model_from_dp

train_data = load_dp_frame()


def train_model():
    """DecisionTree(max_depth=6) on full training CSV — same as FastAPI MLPredictor."""
    clf, le, _feature_cols = train_model_from_dp(train_data)
    return clf, le


In [246]:
class AgentState(TypedDict):
    symptoms: List[str]
    raw_input: str
    feature_vector: List[int]
    predictions: List[dict]
    disease_info: List[dict]
    llm_analysis: Optional[str]
    report: str

In [247]:
def symptom_collector(state: AgentState) -> AgentState:
    valid = [s for s in state["symptoms"] if s in SYMPTOMS]
    print(f"[symptom_collector] Collected {len(valid)} symptoms")
    return {**state, "symptoms": valid}

In [248]:
def nlp_parser(state: AgentState) -> AgentState:
    """Use GPT-4o to extract symptoms from natural language."""
    raw = state.get("raw_input", "")
    if not raw.strip():
        print("[nlp_parser] No text input, skipping")
        return state

    system = (
        "Extract medical symptoms from the patient description. "
        f"Return ONLY a JSON array from: {SYMPTOMS}. "
        "No other text."
    )
    result = call_gpt4o(system, raw)
    try:
        extracted = json.loads(
            result.strip().removeprefix("```json").removesuffix("```")
        )
        new_symptoms = list(set(state["symptoms"] + extracted))
        print(f"[nlp_parser] GPT-4o extracted: {extracted}")
        return {**state, "symptoms": new_symptoms}
    except Exception as e:
        print(f"[nlp_parser] Parse error: {e}")
        return state

In [249]:
def feature_extractor(state: AgentState) -> AgentState:
    vec = [1 if s in state["symptoms"] else 0 for s in SYMPTOMS]
    print(f"[feature_extractor] Vector: {vec}")
    return {**state, "feature_vector": vec}

In [250]:
def ml_predictor(state: AgentState) -> AgentState:
    clf, le = train_model()
    #X = np.array(state["feature_vector"]).reshape(1, -1)
    X = np.array(state["feature_vector"]).reshape(1, -1)

    proba = clf.predict_proba(X)[0]
    top_idx = np.argsort(proba)[::-1][:3]
    preds = [
        {
            "disease_id": le.inverse_transform([i])[0],
            "probability": float(proba[i]),
        }
        for i in top_idx
        if proba[i] > 0.01
    ]
    print(f"[ml_predictor] Top predictions: {preds}")
    return {**state, "predictions": preds}

In [251]:
def db_lookup(state: AgentState) -> AgentState:
    conn = init_db()
    c = conn.cursor()
    info = []
    for pred in state["predictions"]:
        c.execute(
            "SELECT * FROM diseases WHERE id=?",
            (pred["disease_id"],),
        )
        row = c.fetchone()
        if row:
            info.append({
                "id": row[0], "name": row[1],
                "description": row[2], "severity": row[3],
                "treatment": row[4], "prevention": row[5],
                "see_doctor": row[6],
                "probability": pred["probability"],
            })
    conn.close()
    print(f"[db_lookup] Found {len(info)} disease records")
    return {**state, "disease_info": info}

In [252]:
def llm_analyst(state: AgentState) -> AgentState:
    """Use GPT-4o to analyze predictions and provide guidance."""
    disease_summary = "\n".join(
        f"- {d['name']} ({d['probability']*100:.1f}%)"
        for d in state["disease_info"]
    )
    system = (
        "You are a medical AI assistant. Analyze the ML "
        "predictions and provide a helpful summary. "
        "Always note you are NOT a replacement for "
        "professional medical advice. Be concise."
    )
    user_msg = (
        f"Symptoms: {', '.join(state['symptoms'])}\n"
        f"ML Predictions:\n{disease_summary}\n\n"
        "Provide: 1) Why top prediction fits, "
        "2) Warning signs, 3) Recommended next steps."
    )
    analysis = call_gpt4o(system, user_msg)
    print(f"[llm_analyst] Generated analysis ({len(analysis)} chars)")
    return {**state, "llm_analysis": analysis}


In [253]:
def reporter(state: AgentState) -> AgentState:
    lines = ["\n=== Disease Prediction Report ===\n"]
    for i, d in enumerate(state["disease_info"], 1):
        lines.append(f"{i}. {d['name']} ({d['probability']*100:.1f}%)")
        lines.append(f"   Severity: {d['severity']}")
        lines.append(f"   Treatment: {d['treatment']}\n")
    if state.get("llm_analysis"):
        lines.append("\n── GPT-4o AI Analysis ──")
        lines.append(state["llm_analysis"])
    report = "\n".join(lines)
    #print(report)
    return {**state, "report": report}

In [254]:
def build_graph():
    g = StateGraph(AgentState)

    g.add_node("symptom_collector", symptom_collector)
    g.add_node("nlp_parser", nlp_parser)         # GPT-4o
    g.add_node("feature_extractor", feature_extractor)
    g.add_node("ml_predictor", ml_predictor)
    g.add_node("db_lookup", db_lookup)
    g.add_node("llm_analyst", llm_analyst)        # GPT-4o
    g.add_node("reporter", reporter)

    g.set_entry_point("symptom_collector")
    g.add_edge("symptom_collector", "nlp_parser")
    g.add_edge("nlp_parser", "feature_extractor")
    g.add_edge("feature_extractor", "ml_predictor")
    g.add_edge("ml_predictor", "db_lookup")
    g.add_edge("db_lookup", "llm_analyst")
    g.add_edge("llm_analyst", "reporter")
    g.add_edge("reporter", END)

    return g.compile()

In [255]:
if __name__ == "__main__":
    # Set your API key:
    # export OPENAI_API_KEY="sk-..."

    agent = build_graph()
    result = agent.invoke({
        "symptoms": ["fever", "cough", "loss_of_appetite"],
        "raw_input": "I have body aches, chills,  and I'm very tired",
        "feature_vector": [],
        "predictions": [],
        "disease_info": [],
        "llm_analysis": None,
        "report": "",
    })
    print(result["report"])

[symptom_collector] Collected 3 symptoms
[nlp_parser] GPT-4o extracted: ['body_pain', 'chills', 'fatigue']
[feature_extractor] Vector: [1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0]
[ml_predictor] Top predictions: [{'disease_id': 'Malaria', 'probability': 0.7761194029850746}, {'disease_id': 'Typhoid', 'probability': 0.1791044776119403}, {'disease_id': 'Pneumonia', 'probability': 0.04477611940298507}]
[db_lookup] Found 0 disease records


C:\Users\venk2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


[llm_analyst] Generated analysis (911 chars)

=== Disease Prediction Report ===


── GPT-4o AI Analysis ──
Based on the symptoms you've listed—loss of appetite, cough, chills, body pain, fever, and fatigue—the top prediction likely indicates a respiratory infection, such as influenza or COVID-19. 

1) **Why top prediction fits**: The combination of fever, cough, chills, and body pain is common in viral infections, which often lead to systemic symptoms like fatigue and loss of appetite.

2) **Warning signs**: Seek immediate medical attention if you experience difficulty breathing, persistent chest pain, confusion, or bluish lips/face, as these could indicate a more severe condition.

3) **Recommended next steps**: It’s advisable to consult a healthcare professional for a proper diagnosis and treatment. Rest, stay hydrated, and consider over-the-counter medications for symptom relief, but do not delay seeking medical advice.

**Note**: This information is not a replacement for profession